In [ ]:
import google.colab
google.colab.drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip /content/drive/MyDrive/Colab\ Notebooks/dataset.zip -d dataset

Streaming output truncated to the last 5000 lines.
  inflating: dataset/train/train/bor.jpg  
  inflating: dataset/train/train/bov.jpg  
  inflating: dataset/train/train/bow.jpg  
  inflating: dataset/train/train/boy.jpg  
  inflating: dataset/train/train/bpf.jpg  
  inflating: dataset/train/train/bpi.jpg  
  inflating: dataset/train/train/bpj.jpg  
  inflating: dataset/train/train/bpk.jpg  
  inflating: dataset/train/train/bpn.jpg  
  inflating: dataset/train/train/bpp.jpg  
  inflating: dataset/train/train/bpt.jpg  
  inflating: dataset/train/train/bpx.jpg  
  inflating: dataset/train/train/bpz.jpg  
  inflating: dataset/train/train/bqc.jpg  
  inflating: dataset/train/train/bqh.jpg  
  inflating: dataset/train/train/bqi.jpg  
  inflating: dataset/train/train/bqk.jpg  
  inflating: dataset/train/train/bqn.jpg  
  inflating: dataset/train/train/bqp.jpg  
  inflating: dataset/train/train/bqq.jpg  
  inflating: dataset/train/train/bqu.jpg  
  inflating: dataset/train/train/brb.jpg  
  i

In [ ]:
import os
import random
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, random_split
from torchvision import transforms, models

import wandb

from collections import Counter

random.seed(42)
torch.manual_seed(42)

In [ ]:
class ArtistDataset(Dataset):
  def __init__(self, csv, img_dir, transform=None):
    self.data = pd.read_csv(csv)
    self.img_dir = img_dir
    self.transform = transform
    self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(sorted(self.data.iloc[:, 1].unique()))}

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    row = self.data.iloc[idx]
    img_path = os.path.join(self.img_dir, row['id'] + '.jpg')
    image = Image.open(img_path).convert('RGB')
    label = torch.tensor(self.class_to_idx[row['artist']], dtype=torch.long)

    if self.transform:
      image = self.transform(image)

    return image, label


In [ ]:
train_transform = transforms.Compose([
  transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
  transforms.RandomHorizontalFlip(),
  transforms.RandomRotation(15),
  transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
  transforms.Resize((224,224)),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])


# CNN

In [ ]:
class CNN(nn.Module):
  def __init__(self, num_classes):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3, 32, 3, 1, 1),
        nn.BatchNorm2d(32),
        nn.ReLU(),

        nn.Conv2d(32, 64, 3, 1, 1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(64, 128, 3, 1, 1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(128, 256, 3, 1, 1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(256*28*28, 1024),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(1024, num_classes),
    )

  def forward(self, x):
    x = self.features(x)
    x = self.classifier(x)
    return x

# ResNet18

In [ ]:
class ResNetBlock(nn.Module):
  def __init__(self, in_channels, out_channels, stride=1):
    super().__init__()
    self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu = nn.ReLU()
    self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1)
    self.bn2 = nn.BatchNorm2d(out_channels)

    self.shortcut = nn.Sequential()
    if stride != 1 or in_channels != out_channels:
      self.shortcut = nn.Sequential(
          nn.Conv2d(in_channels, out_channels, 1, stride),
          nn.BatchNorm2d(out_channels)
      )

  def forward(self, x):
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)
    out = self.conv2(out)
    out = self.bn2(out)
    out += self.shortcut(x)
    out = self.relu(out)
    return out

In [ ]:
class ResNet18(nn.Module):
  def __init__(self, num_classes):
    super(ResNet18, self).__init__()
    self.in_channels = 64
    self.conv1 = nn.Conv2d(3, 64, 3, 1, 1)
    self.bn1 = nn.BatchNorm2d(64)
    self.relu = nn.ReLU()
    self.maxpool = nn.MaxPool2d(3, 2, 1)

    self.layer1 = self._make_layer(ResNetBlock, 64, 2, 1)
    self.layer2 = self._make_layer(ResNetBlock, 128, 2, 2)
    self.layer3 = self._make_layer(ResNetBlock, 256, 2, 2)
    self.layer4 = self._make_layer(ResNetBlock, 512, 2, 2)

    self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
    self.fc = nn.Linear(512, num_classes)


  def _make_layer(self, block, out_channels, num_blocks, stride):
    strides = [stride] + [1] * (num_blocks - 1)
    layers = []
    for stride in strides:
      layers.append(block(self.in_channels, out_channels, stride))
      self.in_channels = out_channels

    return nn.Sequential(*layers)

  def forward(self, x):
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)
    out = self.maxpool(out)

    out = self.layer1(out)
    out = self.layer2(out)
    out = self.layer3(out)
    out = self.layer4(out)

    out = self.avgpool(out)
    out = out.view(out.size(0), -1)
    out = self.fc(out)
    return out

# ResNet50

In [ ]:
class Bottleneck(nn.Module):
  expansion = 4

  def __init__(self, in_channels, mid_channels, stride=1, downsample=None):
    super().__init__()
    self.conv1 = nn.Conv2d(in_channels, mid_channels, 1, bias=False)
    self.bn1 = nn.BatchNorm2d(mid_channels)

    self.conv2 = nn.Conv2d(mid_channels, mid_channels, 3, stride, 1, bias=False)
    self.bn2 = nn.BatchNorm2d(mid_channels)

    self.conv3 = nn.Conv2d(mid_channels, mid_channels*self.expansion, 1, bias=False)
    self.bn3 = nn.BatchNorm2d(mid_channels*self.expansion)

    self.relu = nn.ReLU()
    self.downsample = downsample


  def forward(self, x):
    identity = x
    out = self.relu(self.bn1(self.conv1(x)))
    out = self.relu(self.bn2(self.conv2(out)))
    out = self.bn3(self.conv3(out))

    if self.downsample is not None:
      identity = self.downsample(x)

    out += identity
    out = self.relu(out)
    return out

In [ ]:
def make_layer(in_channels, mid_channels, blocks, stride):
  layers = []
  downsample = None
  if stride != 1 or in_channels != mid_channels * Bottleneck.expansion:
    downsample = nn.Sequential(
        nn.Conv2d(in_channels, mid_channels*Bottleneck.expansion, 1, stride),
        nn.BatchNorm2d(mid_channels*Bottleneck.expansion),
    )

  layers.append(Bottleneck(in_channels, mid_channels, stride, downsample))
  for _ in range(blocks-1):
    layers.append(Bottleneck(mid_channels*Bottleneck.expansion, mid_channels))

  return nn.Sequential(*layers)

In [ ]:
class ResNet50(nn.Module):
  def __init__(self, num_classes=50):
    super().__init__()
    self.in_channels = 64
    self.conv1 = nn.Conv2d(3, 64, 7, 2, 3, bias=False)
    self.bn1 = nn.BatchNorm2d(64)
    self.relu = nn.ReLU(inplace=True)
    self.maxpool = nn.MaxPool2d(3, 2, 1)

    self.layer1 = make_layer(64, 64, 3, 1)
    self.layer2 = make_layer(256, 128, 4, 2)
    self.layer3 = make_layer(512, 256, 6, 2)
    self.layer4 = make_layer(1024, 512, 3, 2)

    self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
    self.fc = nn.Linear(512*Bottleneck.expansion, num_classes)

  def forward(self, x):
    x = self.relu(self.bn1(self.conv1(x)))
    x = self.maxpool(x)
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)
    x = self.layer4(x)
    x = self.avgpool(x)
    x = x.view(x.size(0), -1)
    x = self.fc(x)
    return x

# DenseNet101

In [ ]:
class _DenseLayer(nn.Module):
  def __init__(self, in_channels, growth_rate, bn_size=4, drop_rate=0):
    super(_DenseLayer, self).__init__()
    self.bn1 = nn.BatchNorm2d(in_channels)
    self.relu1 = nn.ReLU(inplace=True)
    self.conv1 = nn.Conv2d(in_channels, bn_size*growth_rate, 1, 1, bias=False)

    self.bn2 = nn.BatchNorm2d(bn_size*growth_rate)
    self.relu2 = nn.ReLU(inplace=True)
    self.conv2 = nn.Conv2d(bn_size*growth_rate, growth_rate, 3, 1, 1, bias=False)

    self.drop_rate = drop_rate

  def forward(self, x):
    new_features = self.conv1(self.relu1(self.bn1(x)))
    new_features = self.conv2(self.relu2(self.bn2(new_features)))

    if self.drop_rate > 0:
      new_features = F.dropout(new_features, p=self.drop_rate, training=self.training)

    return torch.cat([x, new_features], 1)

In [ ]:
class _DenseBlock(nn.Module):
  def __init__(self, num_layers, in_channels, bn_size, growth_rate, drop_rate):
    super(_DenseBlock, self).__init__()
    layers = []

    for i in range(num_layers):
      layers.append(_DenseLayer(in_channels+i*growth_rate, growth_rate, bn_size, drop_rate))
    self.layers = nn.ModuleList(layers)

  def forward(self, x):
    for layer in self.layers:
      x = layer(x)

    return x

In [ ]:
class _Transition(nn.Module):
  def __init__(self, in_channels, out_channels):
    super(_Transition, self).__init__()
    self.bn = nn.BatchNorm2d(in_channels)
    self.relu = nn.ReLU(inplace=True)
    self.conv = nn.Conv2d(in_channels, out_channels, 1, 1, bias=False)
    self.avgpool = nn.AvgPool2d(2, stride=2)

  def forward(self, x):
    x = self.conv(self.relu(self.bn(x)))
    x = self.avgpool(x)
    return x

In [ ]:
class DenseNet101(nn.Module):
  def __init__(self, growth_rate=32, block_config=(6, 12, 24, 16), num_init_features=64, bn_size=4, drop_rate=0, num_classes=50):
    super(DenseNet101, self).__init__()

    self.features = nn.Sequential(
        nn.Conv2d(3, num_init_features, 7, 2, 3, bias=False),
        nn.BatchNorm2d(num_init_features),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(3, 2, 1),
    )

    num_features = num_init_features
    for i, num_layers in enumerate(block_config):
      block = _DenseBlock(num_layers, num_features, bn_size, growth_rate, drop_rate)
      self.features.add_module('denseblock%d' % (i + 1), block)
      num_features = num_features + num_layers * growth_rate
      if i != len(block_config) - 1:
        trans = _Transition(num_features, num_features // 2)
        self.features.add_module('transition%d' % (i + 1), trans)
        num_features = num_features // 2

    self.features.add_module('norm5', nn.BatchNorm2d(num_features))

    self.classifier = nn.Linear(num_features, num_classes)

    for m in self.modules():
      if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight)
      elif isinstance(m, nn.BatchNorm2d) or isinstance(m, nn.GroupNorm):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)
      elif isinstance(m, nn.Linear):
        nn.init.constant_(m.bias, 0)


  def forward(self, x):
    features = self.features(x)
    out = F.relu(features, inplace=True)
    out = F.adaptive_avg_pool2d(out, (1, 1))
    out = torch.flatten(out, 1)
    out = self.classifier(out)
    return out

# EfficientNet

In [ ]:
class Swish(nn.Module):
  def forward(self, x):
    return x * torch.sigmoid(x)

In [ ]:
class SEBlock(nn.Module):
  def __init__(self, in_channels, reduction=4):
    super().__init__()
    reduced = in_channels // reduction

    self.fc1 = nn.Conv2d(in_channels, reduced, 1)
    self.fc2 = nn.Conv2d(reduced, in_channels, 1)

  def forward(self, x):
    scale = x.mean((2, 3), keepdim=True)
    scale = F.silu(self.fc1(scale))
    scale = torch.sigmoid(self.fc2(scale))
    return x * scale

In [ ]:
class MBConv(nn.Module):
  def __init__(self, in_channels, out_channels, expand_ratio, stride) -> None:
    super().__init__()

    hidden_dim = in_channels * expand_ratio
    self.use_residual = (stride == 1 and in_channels == out_channels)

    layers = []

    if expand_ratio != 1:
      layers += [
          nn.Conv2d(in_channels, hidden_dim, 1, bias = False),
          nn.BatchNorm2d(hidden_dim),
          nn.SiLU()
      ]

    layers += [
        nn.Conv2d(hidden_dim, hidden_dim, 3, stride, 1, groups=hidden_dim, bias=False),
        nn.BatchNorm2d(hidden_dim),
        nn.SiLU(),
    ]

    layers += [SEBlock(hidden_dim)]

    layers += [
        nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
        nn.BatchNorm2d(out_channels)
    ]

    self.block = nn.Sequential(*layers)

  def forward(self, x):
    out = self.block(x)
    if self.use_residual:
      return x + out
    return out


In [ ]:
class EfficientNet(nn.Module):
  def __init__(self, num_classes=50):
    super().__init__()

    self.stem = nn.Sequential(
        nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
        nn.BatchNorm2d(32),
        nn.SiLU()
    )

    config = [
        (1, 16, 1, 1),
        (6, 24, 2, 2),
        (6, 40, 2, 2),
        (6, 80, 3, 2),
        (6, 112, 3, 1),
        (6, 192, 4, 2),
        (6, 320, 1, 1)
    ]

    layers = []
    in_channels = 32

    for expaand, out, repeats, stride in config:
      layers.append(MBConv(in_channels, out, expaand, stride))
      for _ in range(repeats-1):
        layers.append(MBConv(out, out, expaand, 1))
      in_channels = out

    self.blocks = nn.Sequential(*layers)

    self.head = nn.Sequential(
        nn.Conv2d(in_channels, 1280, 1, bias=False),
        nn.BatchNorm2d(1280),
        nn.SiLU(),
    )

    self.classifier = nn.Linear(1280, num_classes)

  def forward(self, x):
    x = self.stem(x)
    x = self.blocks(x)
    x = self.head(x)
    x = x.mean([2, 3])
    return self.classifier(x)

# W&B


In [ ]:
run = wandb.init(
    project='artist_classification',
    config={
        'learning_rate': 0.005,
        'batch_size': 64,
        'epochs': 50,
        'architecture': 'EfficientNet',
    }
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: pfrsam (pfrsam-hanyang-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Data Augmentation

In [ ]:
augmentation = transforms.Compose([
  transforms.RandomRotation(degrees=(-20, 20)),
  transforms.RandomHorizontalFlip(p=0.5),
  transforms.RandomResizedCrop(224, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
  transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
  transforms.RandomApply([
      transforms.GaussianBlur(kernel_size=3),
      transforms.RandomAdjustSharpness(sharpness_factor=2),
  ], p=0.5),
])

In [ ]:
def balance_dataset(csv_path, img_dir, target_count=150):
    df = pd.read_csv(csv_path)

    counts = df['artist'].value_counts()
    print("Before balancing:\n", counts)

    new_rows = []

    for artist, count in counts.items():
        if count < target_count:
            needed = target_count - count
            print(f"Augmenting {artist}: {count} → {target_count} ({needed} new images)")

            artist_images = df[df['artist'] == artist]['id'].tolist()

            for i in range(needed):
                original_name = random.choice(artist_images)
                img_path = os.path.join(img_dir, original_name + ".jpg")

                img = Image.open(img_path).convert("RGB")
                aug_img = augmentation(img)

                new_filename = f"{original_name}_aug_{i}"
                save_path = os.path.join(img_dir, new_filename + ".jpg")
                aug_img.save(save_path)

                new_rows.append({"id": new_filename, "artist": artist})

    if new_rows:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
        df.to_csv(csv_path, index=False)
        print("\nDataset balancing complete.")
    else:
        print("\nDataset is already balanced.")

    print("After balancing:\n", df['artist'].value_counts())



In [ ]:
balance_dataset(
    csv_path="dataset/train.csv",
    img_dir="dataset/train/train",
    target_count=300
)

Before balancing:
 artist
Vincent van Gogh             560
Edgar Degas                  443
Pablo Picasso                274
Pierre-Auguste Renoir        210
Albrecht Du rer              201
Paul Gauguin                 195
Francisco Goya               176
Rembrandt                    161
Marc Chagall                 156
Titian                       153
Alfred Sisley                147
Paul Klee                    124
Amedeo Modigliani            121
Rene Magritte                121
Andy Warhol                  118
Sandro Botticelli            111
Henri Matisse                111
Mikhail Vrubel               106
Hieronymus Bosch             105
Leonardo da Vinci             96
Salvador Dali                 88
Peter Paul Rubens             83
Kazimir Malevich              82
Pieter Bruegel                79
Frida Kahlo                   72
Diego Velazquez               71
Andrei Rublev                 70
Joan Miro                     69
Raphael                       68
Gustav Klimt     

In [ ]:
train_dataset = ArtistDataset(csv='dataset/train.csv', img_dir='dataset/train/train', transform=train_transform)
val_dataset = ArtistDataset(csv='dataset/train.csv', img_dir='dataset/train/train', transform=val_transform)

labels = train_dataset.data['artist'].values
train_idx, val_idx = [], []

val_ratio = 0.2

for c in np.unique(labels):
  idx = np.where(labels == c)[0]
  idx = torch.tensor(idx)[torch.randperm(len(idx))]
  split = int(len(idx) * val_ratio)
  val_idx.extend(idx[:split].tolist())
  train_idx.extend(idx[split:].tolist())

train_subset = torch.utils.data.Subset(train_dataset, train_idx)
val_subset = torch.utils.data.Subset(val_dataset, val_idx)

class_counts = train_dataset.data['artist'].value_counts()
class_weights = 1. / torch.tensor([class_counts[train_dataset.data.iloc[i]['artist']] for i in train_idx], dtype=torch.float)
train_sampler = WeightedRandomSampler(class_weights, num_samples=len(train_subset), replacement=True)

train_loader = DataLoader(train_subset, batch_size=64, sampler=train_sampler, num_workers=10)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False, num_workers=10)


# Train


In [ ]:
def mixup_data(x, y, alpha=0.4):
  lam = torch.distributions.Beta(alpha, alpha).sample().to(x.device) if alpha > 0 else 1
  batch_size = x.size(0)
  index = torch.randperm(batch_size).to(x.device)
  mixed_x = lam * x + (1 - lam) * x[index]
  y_a, y_b = y, y[index]
  return mixed_x, y_a, y_b, lam

In [ ]:
def train(model, epochs=50, alpha_mixup=0.4):
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model.to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
  scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

  best_val_acc = 0.0
  patience_counter = 0

  for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
      images, labels = images.to(device), labels.to(device)
      inputs, targets_a, targets_b, lam = mixup_data(images, labels, alpha_mixup)

      optimizer.zero_grad()
      outputs = model(inputs)
      loss = lam*criterion(outputs, targets_a) + (1-lam)*criterion(outputs, targets_b)
      loss.backward()
      optimizer.step()

      running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    val_corrects = 0
    with torch.no_grad():
      for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        val_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, 1)
        val_corrects += torch.sum(preds == labels)

    val_loss /= len(val_loader.dataset)
    val_acc = val_corrects.double() / len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    wandb.log({
      "epoch": epoch+1,
      "train_loss": epoch_loss,
      "val_loss": val_loss,
      "val_accuracy": val_acc
    })

    scheduler.step(val_loss)

    if val_acc > best_val_acc:
      best_val_acc = val_acc
      patience_counter = 0
      torch.save(model.state_dict(), "best_model.pth")
    else:
      patience_counter += 1
      if patience_counter > 100:
        print("Early stopping triggered.")
        break

  model.load_state_dict(torch.load("best_model.pth"))
  return model


In [ ]:
def validate(model, data_loader, loss_fn):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()

    running_loss = 0.0
    running_corrects = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss_value = loss_fn(outputs, labels)

            running_loss += loss_value.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == labels)
            total_samples += images.size(0)

    avg_loss = running_loss / total_samples
    avg_acc = running_corrects.double() / total_samples

    return avg_loss, avg_acc


In [ ]:
loss_fn = nn.CrossEntropyLoss()

# Test

In [ ]:
class ArtistTestDataset(Dataset):
    def __init__(self, csv, img_dir, transform=None):
        self.data = pd.read_csv(csv)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        filename = self.data.iloc[idx, 0]

        img_path = os.path.join(self.img_dir, filename + '.jpg')
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, filename


In [ ]:
idx_to_class = {idx: cls_name for cls_name, idx in train_dataset.class_to_idx.items()}

In [ ]:
def predict_test(model, data_loader, csv_path='submission.csv'):
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model.to(device)
  model.eval()

  results = []

  with torch.no_grad():
      for images, filenames in data_loader:
          images = images.to(device)

          outputs = model(images)
          _, preds = torch.max(outputs, 1)

          for fname, pred_idx in zip(filenames, preds.cpu().numpy()):
              artist_name = idx_to_class[pred_idx]
              results.append({"id": fname, "artist": artist_name})

  df = pd.DataFrame(results)
  df.to_csv(csv_path, index=False)

  print(f"Submission with names saved as {csv_path}")

In [ ]:
test_dataset = ArtistTestDataset(
    csv='dataset/test.csv',
    img_dir='dataset/test/test',
    transform=val_transform)

test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

loss_fn = nn.CrossEntropyLoss()

In [ ]:
model = train(ResNet18(num_classes=50), epochs=50)

Epoch 1/50 | Train Loss: 3.6872 | Val Loss: 3.6247 | Val Acc: 0.1201
Epoch 2/50 | Train Loss: 3.4741 | Val Loss: 3.3392 | Val Acc: 0.1386
Epoch 3/50 | Train Loss: 3.3889 | Val Loss: 3.1582 | Val Acc: 0.1883
Epoch 4/50 | Train Loss: 3.2883 | Val Loss: 3.0534 | Val Acc: 0.2049
Epoch 5/50 | Train Loss: 3.1805 | Val Loss: 2.9650 | Val Acc: 0.2370
Epoch 6/50 | Train Loss: 3.0821 | Val Loss: 2.7651 | Val Acc: 0.2675
Epoch 7/50 | Train Loss: 3.0428 | Val Loss: 2.7557 | Val Acc: 0.2727
Epoch 8/50 | Train Loss: 3.0078 | Val Loss: 2.6090 | Val Acc: 0.2873
Epoch 9/50 | Train Loss: 2.9507 | Val Loss: 2.7862 | Val Acc: 0.2792
Epoch 10/50 | Train Loss: 2.9258 | Val Loss: 2.7212 | Val Acc: 0.3036
Epoch 11/50 | Train Loss: 2.8133 | Val Loss: 2.5912 | Val Acc: 0.3192
Epoch 12/50 | Train Loss: 2.7477 | Val Loss: 2.4028 | Val Acc: 0.3695
Epoch 13/50 | Train Loss: 2.7907 | Val Loss: 2.4215 | Val Acc: 0.3481
Epoch 14/50 | Train Loss: 2.6803 | Val Loss: 2.2928 | Val Acc: 0.3997
Epoch 15/50 | Train Loss: 2.5

In [ ]:
avg_loss, avg_acc = validate(model, val_loader, loss_fn)

In [ ]:
print(f'Loss: {avg_loss}. Accuracy: {avg_acc}')

Loss: 0.4808108472204828. Accuracy: 0.8974025974025974


In [ ]:
predict_test(model, test_loader, csv_path='resnet18-5-6.csv')

Submission with names saved as resnet18-5-6.csv


In [ ]:
model = train(ResNet50(num_classes=50), epochs=200)

Epoch 1/200 | Train Loss: 3.9929 | Val Loss: 5.2243 | Val Acc: 0.0686
Epoch 2/200 | Train Loss: 3.6760 | Val Loss: 3.5687 | Val Acc: 0.0863


KeyboardInterrupt: 

In [ ]:
avg_loss, avg_acc = validate(model, val_loader, loss_fn)

In [ ]:
print(f'Loss: {avg_loss}. Accuracy: {avg_acc}')

In [ ]:
predict_test(model, test_loader, csv_path='resnet50-2')

In [ ]:
model = train(DenseNet101(num_classes=50), epochs=200)

Epoch 1/200 | Train Loss: 3.9452 | Val Loss: 3.6931 | Val Acc: 0.0922
Epoch 2/200 | Train Loss: 3.7376 | Val Loss: 3.7227 | Val Acc: 0.0725
Epoch 3/200 | Train Loss: 3.6735 | Val Loss: 3.5915 | Val Acc: 0.0667
Epoch 4/200 | Train Loss: 3.6033 | Val Loss: 3.5273 | Val Acc: 0.1039
Epoch 5/200 | Train Loss: 3.5064 | Val Loss: 4.5920 | Val Acc: 0.0882
Epoch 6/200 | Train Loss: 3.4262 | Val Loss: 3.5483 | Val Acc: 0.0824
Epoch 7/200 | Train Loss: 3.3998 | Val Loss: 3.1975 | Val Acc: 0.1490
Epoch 8/200 | Train Loss: 3.3309 | Val Loss: 3.3501 | Val Acc: 0.1275
Epoch 9/200 | Train Loss: 3.2705 | Val Loss: 3.6846 | Val Acc: 0.1275
Epoch 10/200 | Train Loss: 3.2748 | Val Loss: 4.6051 | Val Acc: 0.0961
Epoch 11/200 | Train Loss: 3.2050 | Val Loss: 3.1577 | Val Acc: 0.1588
Epoch 12/200 | Train Loss: 3.1258 | Val Loss: 3.1466 | Val Acc: 0.1784
Epoch 13/200 | Train Loss: 3.1926 | Val Loss: 3.1692 | Val Acc: 0.1706
Epoch 14/200 | Train Loss: 3.1540 | Val Loss: 3.2061 | Val Acc: 0.1451
Epoch 15/200 | 

KeyboardInterrupt: 

In [ ]:
avg_loss, avg_acc = validate(model, val_loader, loss_fn)

In [ ]:
print(f'Loss: {avg_loss}. Accuracy: {avg_acc}')

In [ ]:
predict_test(model, test_loader, csv_path='densenet101-2.csv')

In [ ]:
model = train(EfficientNet(num_classes=50) epochs=200)

In [ ]:
avg_loss, avg_acc = validate(model, val_loader, loss_fn)

In [ ]:
print(f'Loss: {avg_loss}. Accuracy: {avg_acc}')

In [ ]:
predict_test(model, test_loader, csv_path='efficientnet.csv')